# Semantic SASRec (Hierarchical Semantic IDs)

**Part 2c of the Sequential Recommenders series**

Each item is **4 tokens** instead of 1. The model predicts the next item's Semantic ID level by level:
- Level 0: predicted from sequence context alone
- Level 1: conditioned on Level 0 (via teacher forcing at 90% during training)
- Level 2: conditioned on Levels 0 + 1
- Level 3: conditioned on Levels 0 + 1 + 2

**Prerequisites:** Run the RQVAE notebook first (checkpoints 1-2), then the SASRec notebook (checkpoint 3).


---
## 1. Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import gc
import ast
import gzip
import json
import time
import pickle
import subprocess
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict, NamedTuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from tqdm.notebook import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

%matplotlib inline
sns.set_style('whitegrid')

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")
if device == 'cuda':
    torch.set_float32_matmul_precision('high')
    print(f"GPU: {torch.cuda.get_device_name()}")

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: cuda
GPU: NVIDIA A10G


---
## Load Checkpoints

Load pre-computed data from the RQVAE and SASRec notebooks.


In [2]:
# ============================================================
# Load Checkpoint 1 (run this instead of cells 2-6 to resume)
# ============================================================
CHECKPOINT_DIR = 'checkpoints'

item_embeddings = torch.load(f'{CHECKPOINT_DIR}/item_embeddings.pt', weights_only=True)
with open(f'{CHECKPOINT_DIR}/data_checkpoint1.pkl', 'rb') as f:
    _ckpt = pickle.load(f)
    games_df = _ckpt['games_df']
    user_sequences = _ckpt['user_sequences']
    sasrec_game_ids = _ckpt['sasrec_game_ids']
    del _ckpt

print(f'Checkpoint 1 loaded: item_embeddings {item_embeddings.shape}, '
      f'{len(games_df)} games, {len(user_sequences)} users')

Checkpoint 1 loaded: item_embeddings torch.Size([5232, 1024]), 5232 games, 56808 users


In [3]:
# ============================================================
# Load Checkpoint 2 (run this + checkpoint 1 load to skip RQVAE)
# ============================================================
CHECKPOINT_DIR = 'checkpoints'

with open(f'{CHECKPOINT_DIR}/data_checkpoint2.pkl', 'rb') as f:
    _ckpt = pickle.load(f)
    semantic_ids_3 = _ckpt['semantic_ids_3']
    semantic_ids_4 = _ckpt['semantic_ids_4']
    semantic_token_ids = _ckpt['semantic_token_ids']
    app_id_to_tokens = _ckpt['app_id_to_tokens']
    CODEBOOK_SIZE = _ckpt['CODEBOOK_SIZE']
    NUM_LEVELS = _ckpt['NUM_LEVELS']
    del _ckpt

print(f'Checkpoint 2 loaded: {len(app_id_to_tokens)} items, '
      f'{NUM_LEVELS} levels x {CODEBOOK_SIZE} codes')

Checkpoint 2 loaded: 5232 items, 4 levels x 256 codes


In [4]:
# ============================================================
# Load Checkpoint 3 (run this + checkpoints 1-2 to skip to training)
# ============================================================
CHECKPOINT_DIR = 'checkpoints'

with open(f'{CHECKPOINT_DIR}/data_checkpoint3.pkl', 'rb') as f:
    _ckpt = pickle.load(f)
    user_seqs = _ckpt['user_seqs']
    item_to_id = _ckpt['item_to_id']
    id_to_item = _ckpt['id_to_item']
    n_items = _ckpt['n_items']
    del _ckpt

MAX_SEQ_LEN = 100
print(f'Checkpoint 3 loaded: {len(user_seqs)} users, {n_items} items')

Checkpoint 3 loaded: 56692 users, 5232 items


---
## 7. Semantic SASRec

Each item is now **4 tokens** instead of 1. The model predicts the next item's Semantic ID *level by level*:
- Level 0: predicted from sequence context alone
- Level 1: conditioned on Level 0 (via teacher forcing at 90% during training)
- Level 2: conditioned on Levels 0 + 1
- Level 3: conditioned on Levels 0 + 1 + 2

Scoring candidates sums log-probabilities across all 4 levels.

In [5]:
# ============================================================
# Semantic SASRec Model
# ============================================================

class SemanticSASRec(nn.Module):
    def __init__(self, vocab_size=1024, num_levels=4, codebook_size=256,
                 max_seq_length=100, input_dim=128, hidden_dim=256,
                 num_heads=4, num_blocks=2, mlp_dim=512, dropout_rate=0.1):
        super().__init__()
        self.num_levels = num_levels
        self.codebook_size = codebook_size
        self.hidden_dim = hidden_dim
        self.input_dim = input_dim
        max_token_len = max_seq_length * num_levels
        
        self.token_emb = nn.Embedding(vocab_size + 1, input_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_token_len + 1, input_dim, padding_idx=0)
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        self.emb_dropout = nn.Dropout(dropout_rate)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=mlp_dim, dropout=dropout_rate,
            activation='relu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_blocks)
        self.ln_f = nn.LayerNorm(hidden_dim, eps=1e-8)
        
        # Level-specific prediction heads
        self.level_heads = nn.ModuleList([nn.Linear(hidden_dim, codebook_size) for _ in range(num_levels)])
        # Context combiners for levels 1-3
        self.context_combiners = nn.ModuleList([nn.Linear(hidden_dim * 2, hidden_dim) for _ in range(num_levels - 1)])
        
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)
            if m.padding_idx is not None:
                with torch.no_grad(): m.weight[m.padding_idx].fill_(0)
    
    def forward(self, input_ids):
        B, T = input_ids.size()
        embs = self.token_emb(input_ids) * self.input_dim**0.5
        positions = torch.arange(1, T+1, device=input_ids.device).unsqueeze(0).expand(B, -1)
        positions = positions * (input_ids != 0).long()
        hidden = self.input_projection(self.emb_dropout(embs + self.pos_emb(positions)))
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=input_ids.device)
        hidden = self.transformer(hidden, mask=causal_mask, is_causal=True)
        return self.ln_f(hidden)
    
    def _predict_from_hidden(self, last_hidden, teacher_forcing=True, target_tokens=None):
        """Predict next item's level logits from pre-computed hidden state."""
        predictions = {}
        
        for level in range(self.num_levels):
            if level == 0:
                context = last_hidden
            else:
                if teacher_forcing and target_tokens is not None:
                    prev = target_tokens[:, :level]
                else:
                    prev = self._sample_from_predictions(predictions, level)
                prev_proj = self.input_projection(self.token_emb(prev)).mean(dim=1)
                context = self.context_combiners[level-1](torch.cat([last_hidden, prev_proj], -1))
            
            predictions[f'logits_l{level}'] = self.level_heads[level](context)
        return predictions
    
    def predict_next_item(self, input_ids, teacher_forcing=True, target_tokens=None):
        hidden = self.forward(input_ids)
        return self._predict_from_hidden(hidden[:, -1, :], teacher_forcing, target_tokens)
    
    def _sample_from_predictions(self, predictions, up_to):
        tokens = []
        for l in range(up_to):
            t = predictions[f'logits_l{l}'].argmax(dim=-1)
            tokens.append(l * self.codebook_size + t)
        return torch.stack(tokens, dim=1)
    
    def score_candidates(self, input_ids, candidate_tokens):
        """Score candidates by summing log-probs across all levels."""
        B, C, _ = candidate_tokens.shape
        hidden = self.forward(input_ids)
        last_hidden = hidden[:, -1, :]
        flat = candidate_tokens.view(-1, self.num_levels)
        
        all_scores = []
        for level in range(self.num_levels):
            if level == 0:
                ctx = last_hidden.unsqueeze(1).expand(-1, C, -1).reshape(-1, self.hidden_dim)
            else:
                prev = flat[:, :level]
                prev_proj = self.input_projection(self.token_emb(prev)).mean(dim=1)
                exp_lh = last_hidden.unsqueeze(1).expand(-1, C, -1).reshape(-1, self.hidden_dim)
                ctx = self.context_combiners[level-1](torch.cat([exp_lh, prev_proj], -1))
            
            logits = self.level_heads[level](ctx)
            targets = flat[:, level] % self.codebook_size
            lp = F.log_softmax(logits, dim=-1)
            scores = lp.gather(1, targets.unsqueeze(1)).squeeze(1)
            all_scores.append(scores.view(B, C))
        
        return torch.stack(all_scores, dim=-1).sum(dim=-1)
    
    def training_step(self, input_ids, pos_tokens, neg_tokens, tf_ratio=0.9):
        use_tf = torch.rand(1).item() < tf_ratio
        # Forward pass ONCE, reuse for both pos and neg predictions
        hidden = self.forward(input_ids)
        last_hidden = hidden[:, -1, :]
        pos_pred = self._predict_from_hidden(last_hidden, use_tf, pos_tokens)
        neg_pred = self._predict_from_hidden(last_hidden, use_tf, neg_tokens)
        
        losses = {}
        for level in range(self.num_levels):
            pos_tgt = pos_tokens[:, level] % self.codebook_size
            neg_tgt = neg_tokens[:, level] % self.codebook_size
            
            pos_loss = F.cross_entropy(pos_pred[f'logits_l{level}'], pos_tgt)
            neg_probs = F.softmax(neg_pred[f'logits_l{level}'], dim=-1)
            neg_token_p = neg_probs.gather(1, neg_tgt.unsqueeze(1)).squeeze(1)
            neg_loss = -torch.log(1 - neg_token_p + 1e-8).mean()
            
            losses[f'loss_l{level}'] = pos_loss + neg_loss
        return losses


print('SemanticSASRec model defined')

SemanticSASRec model defined


In [6]:
# ============================================================
# Prepare semantic sequences and train
# ============================================================

# Convert user sequences to token sequences
user_token_seqs = {}
for user, seq in user_seqs.items():
    token_seq = []
    valid = True
    for item_id in seq:
        app_id = id_to_item[item_id]
        if app_id not in app_id_to_tokens:
            valid = False
            break
        token_seq.extend(app_id_to_tokens[app_id])
    if valid and len(seq) >= 3:
        user_token_seqs[user] = (token_seq, len(seq))

print(f'Users with semantic sequences: {len(user_token_seqs):,}')

# Build candidate pool
candidate_pool = list(set(tuple(v) for v in app_id_to_tokens.values()))
candidate_to_idx = {c: i for i, c in enumerate(candidate_pool)}
print(f'Candidate pool: {len(candidate_pool):,} unique items')


def make_semantic_batch(user_token_seqs, users, num_levels=4, max_seq_len=100):
    """Create SemanticSASRec training batch with in-batch negatives."""
    max_token_len = max_seq_len * num_levels
    B = len(users)
    
    input_ids = torch.zeros(B, max_token_len, dtype=torch.long)
    pos_tokens = torch.zeros(B, num_levels, dtype=torch.long)
    neg_tokens = torch.zeros(B, num_levels, dtype=torch.long)
    pos_list = []
    
    for i, user in enumerate(users):
        full_tokens, seq_len = user_token_seqs[user]
        # Use seq[:-2] for training
        train_len = min(seq_len - 2, max_seq_len)
        train_tokens = full_tokens[:train_len * num_levels]
        tl = len(train_tokens)
        input_ids[i, -tl:] = torch.tensor(train_tokens)
        
        # Positive: next item after training sequence
        pos_start = train_len * num_levels
        pos = full_tokens[pos_start:pos_start + num_levels]
        pos_tokens[i] = torch.tensor(pos)
        pos_list.append(pos)
    
    # In-batch negative sampling
    for i in range(B):
        neg_idx = np.random.choice([j for j in range(B) if j != i]) if B > 1 else 0
        neg_tokens[i] = torch.tensor(pos_list[neg_idx])
    
    return input_ids, pos_tokens, neg_tokens


def evaluate_semantic_sasrec(model, user_token_seqs, candidate_pool, mode='val',
                              n_neg=100, num_levels=4, max_seq_len=100, device='cpu'):
    model.eval()
    ranks = []
    users = list(user_token_seqs.keys())
    max_token_len = max_seq_len * num_levels
    
    for batch_start in tqdm(range(0, len(users), 256), desc=f'Eval ({mode})', leave=False):
        batch_users = users[batch_start:batch_start+256]
        batch_data = []
        
        for user in batch_users:
            full_tokens, seq_len = user_token_seqs[user]
            if mode == 'val':
                if seq_len < 3: continue
                split = seq_len - 2
            else:
                if seq_len < 2: continue
                split = seq_len - 1
            
            inp = full_tokens[:split * num_levels]
            tgt = full_tokens[split * num_levels:(split + 1) * num_levels]
            
            # Get seen items for this user
            seen = set()
            for j in range(0, len(full_tokens), num_levels):
                item = tuple(full_tokens[j:j+num_levels])
                if item in candidate_to_idx:
                    seen.add(candidate_to_idx[item])
            
            batch_data.append((inp, tgt, seen))
        
        if not batch_data: continue
        
        ml = min(max(len(d[0]) for d in batch_data), max_token_len)
        input_t = torch.zeros(len(batch_data), ml, dtype=torch.long, device=device)
        cand_t = torch.zeros(len(batch_data), n_neg + 1, num_levels, dtype=torch.long, device=device)
        
        for i, (inp, tgt, seen) in enumerate(batch_data):
            tokens = inp[-ml:] if len(inp) > ml else inp
            input_t[i, -len(tokens):] = torch.tensor(tokens, device=device)
            cand_t[i, 0] = torch.tensor(tgt, device=device)
            
            valid = list(set(range(len(candidate_pool))) - seen)
            neg_idx = np.random.choice(valid, size=min(n_neg, len(valid)), replace=len(valid) < n_neg)
            for j, ni in enumerate(neg_idx):
                cand_t[i, j+1] = torch.tensor(candidate_pool[ni], device=device)
        
        with torch.no_grad():
            scores = model.score_candidates(input_t, cand_t)
            for i in range(len(batch_data)):
                rank = (scores[i, 0] < scores[i]).sum().item() + 1
                ranks.append(rank)
    
    ranks = np.array(ranks)
    return {'hr@10': (ranks <= 10).mean(), 'ndcg@10': np.where(ranks <= 10, 1/np.log2(ranks+1), 0).mean()}


print('Semantic training/evaluation functions defined')

Users with semantic sequences: 56,692
Candidate pool: 5,232 unique items
Semantic training/evaluation functions defined


In [7]:
# ============================================================
# Train Semantic SASRec
# ============================================================

sem_sasrec = SemanticSASRec(
    vocab_size=NUM_LEVELS * CODEBOOK_SIZE,  # 4 * 256 = 1024
    num_levels=NUM_LEVELS,
    codebook_size=CODEBOOK_SIZE,
    max_seq_length=MAX_SEQ_LEN,
    input_dim=128, hidden_dim=256, num_heads=4,
    num_blocks=2, mlp_dim=512, dropout_rate=0.1
).to(device)

print(f'SemanticSASRec params: {sum(p.numel() for p in sem_sasrec.parameters()):,}')

opt_sem = torch.optim.AdamW(sem_sasrec.parameters(), lr=1e-3, betas=(0.9, 0.98))
all_sem_users = list(user_token_seqs.keys())
BATCH_SIZE_SEM = 128
NUM_EPOCHS_SEM = 200

best_sem_ndcg = 0
for epoch in tqdm(range(NUM_EPOCHS_SEM), desc="Semantic SASRec Training"):
    sem_sasrec.train()
    np.random.shuffle(all_sem_users)
    epoch_loss = 0
    n_batches = 0
    
    for i in range(0, len(all_sem_users), BATCH_SIZE_SEM):
        batch_users = all_sem_users[i:i+BATCH_SIZE_SEM]
        inp, pos, neg = make_semantic_batch(user_token_seqs, batch_users, NUM_LEVELS, MAX_SEQ_LEN)
        inp, pos, neg = inp.to(device), pos.to(device), neg.to(device)
        
        opt_sem.zero_grad()
        losses = sem_sasrec.training_step(inp, pos, neg, tf_ratio=0.9)
        total_loss = sum(losses.values()) / len(losses)
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(sem_sasrec.parameters(), 1.0)
        opt_sem.step()
        epoch_loss += total_loss.item()
        n_batches += 1
    
    if (epoch + 1) % 20 == 0:
        val = evaluate_semantic_sasrec(sem_sasrec, user_token_seqs, candidate_pool,
                                       'val', device=device)
        print(f'Epoch {epoch+1:3d} | loss: {epoch_loss/max(n_batches,1):.4f} | '
              f'val HR@10: {val["hr@10"]:.4f} | NDCG@10: {val["ndcg@10"]:.4f}')
        if val['ndcg@10'] > best_sem_ndcg:
            best_sem_ndcg = val['ndcg@10']
            torch.save(sem_sasrec.state_dict(), 'semantic_sasrec_best.pt')

# Final test
sem_sasrec.load_state_dict(torch.load('semantic_sasrec_best.pt', map_location=device, weights_only=True))
test_semantic = evaluate_semantic_sasrec(sem_sasrec, user_token_seqs, candidate_pool,
                                         'test', device=device)
print(f'\nSemantic SASRec Test: HR@10={test_semantic["hr@10"]:.4f}, NDCG@10={test_semantic["ndcg@10"]:.4f}')

SemanticSASRec params: 1,927,424


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Semantic SASRec Training:   0%|          | 0/200 [00:00<?, ?it/s]

Eval (val):   0%|          | 0/222 [00:00<?, ?it/s]

Epoch  20 | loss: 2.4855 | val HR@10: 0.8722 | NDCG@10: 0.6275


...

Epoch 180 | loss: 1.4005 | val HR@10: 0.9651 | NDCG@10: 0.9391


Eval (val):   0%|          | 0/222 [00:00<?, ?it/s]

Epoch 200 | loss: 1.4258 | val HR@10: 0.9649 | NDCG@10: 0.9406


Eval (test):   0%|          | 0/222 [00:00<?, ?it/s]


Semantic SASRec Test: HR@10=0.5616, NDCG@10=0.3581


In [8]:
test_semantic = evaluate_semantic_sasrec(sem_sasrec, user_token_seqs, candidate_pool,
                                         'test', device=device,n_neg=len(candidate_pool)-1)
print(f'\nSemantic SASRec Test: HR@10={test_semantic["hr@10"]:.4f}, NDCG@10={test_semantic["ndcg@10"]:.4f}')

Eval (test):   0%|          | 0/222 [00:00<?, ?it/s]


Semantic SASRec Test: HR@10=0.0967, NDCG@10=0.0566
